# Attention Mechanism

**Module:** 05 — LLM Fundamentals

Query/Key/Value attention, multi-head attention, and cross attention—worked intuitions with numpy.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compute a tiny attention distribution by hand/numpy
- Explain why multi-head attention helps
- Contrast self-attention vs cross-attention
- Connect attention patterns to RAG/context utilization


## Q, K, V

**Definition.** Attention computes weights by comparing **queries** to **keys**, then mixes **values**.

**Why it matters.** It is the information-routing mechanism of Transformers.

**How it works.** For each position: scores = QK^T / sqrt(d) -> softmax -> weighted sum of V.

**Intuition.** Query asks; keys advertise; values deliver content.

**Common pitfalls.**
- Forgetting scaling by sqrt(d)
- Looking at weights as literal human explanations always

**When to use.** Anytime you reason about context use or long prompts.

```mermaid
flowchart LR
  X[Inputs] --> Q[Q]
  X --> K[K]
  X --> V[V]
  Q --> S[Scores QK^T]
  K --> S
  S --> Soft[Softmax]
  Soft --> O[Weighted sum of V]
  V --> O
```


In [ ]:
# Demo 1 — tiny attention
import numpy as np
Q = np.array([[1.0, 0.0], [0.0, 1.0]])
K = np.array([[1.0, 0.0], [0.5, 0.5], [0.0, 1.0]])
V = np.array([[10.0, 0.0], [0.0, 10.0], [0.0, 5.0]])
scale = np.sqrt(Q.shape[-1])
scores = Q @ K.T / scale
w = np.exp(scores - scores.max(axis=1, keepdims=True))
w = w / w.sum(axis=1, keepdims=True)
out = w @ V
print("weights\n", w.round(3))
print("out\n", out.round(3))


In [ ]:
# Demo 2 — causal self-attention scores
T=4; d=8
rng = np.random.default_rng(0)
X = rng.normal(size=(T,d))
Wq=Wk=Wv = rng.normal(size=(d,d))
Q,K,V = X@Wq, X@Wk, X@Wv
scores = Q@K.T/np.sqrt(d)
scores = np.where(np.tril(np.ones((T,T), bool)), scores, -1e9)
w = np.exp(scores - scores.max(1, keepdims=True)); w/=w.sum(1, keepdims=True)
print(w.round(2))


In [ ]:
# Demo 3 — attention as soft retrieval
keys = ["refund policy", "shipping", "password"]
query = "money back"
# bag overlap as score
scores = [len(set(query.split()) & set(k.split())) for k in keys]
print(list(zip(keys, scores)))


### Try it yourself — Q, K, V

1. Compute attention weights for a 2x3 toy by hand and verify with numpy.


## Multi-Head Attention

**Definition.** Split Q/K/V into multiple heads so different heads specialize in different relations.

**Why it matters.** One average attention pattern is too low-capacity for syntax + coreference + position.

**How it works.** Project into H heads, attend per head, concat, project out.

**Intuition.** Several specialists voting, not one generalist.

**Common pitfalls.**
- Assuming more heads always help
- Visualizing one head as the whole model

**When to use.** Standard in virtually all Transformers.


In [ ]:
# Demo 1 — split heads
import numpy as np
T,D,H = 4, 8, 4
x = np.arange(T*D).reshape(T,D)
head_dim = D//H
heads = x.reshape(T,H,head_dim).transpose(1,0,2)
print(heads.shape)  # (H,T,head_dim)


In [ ]:
# Demo 2 — capacity intuition
print({"single_head_patterns": 1, "h8_patterns": 8})


In [ ]:
# Demo 3 — concat heads
H,T,hd = 4, 3, 2
rng = np.random.default_rng(0)
heads = rng.normal(size=(H,T,hd))
concat = heads.transpose(1,0,2).reshape(T, H*hd)
print(concat.shape)


### Try it yourself — Multi-Head Attention

1. Why might syntax and long-range links want different heads?


## Cross Attention

**Definition.** **Cross attention** uses queries from one sequence and keys/values from another (e.g., decoder attending to encoder).

**Why it matters.** Critical in encoder-decoder models and in multimodal fusion patterns.

**How it works.** Q from target side; K,V from source/memory side.

**Intuition.** Writing while looking at a reference document.

**Common pitfalls.**
- Confusing cross-attn with retrieval RAG (related idea, different layer)
- Shape mismatches

**When to use.** Translation/classic seq2seq; some multimodal blocks; prefix architectures.


In [ ]:
# Demo 1 — cross-attn shapes
B, Tt, Ts, D = 1, 3, 5, 16
print({"Q": (B,Tt,D), "K": (B,Ts,D), "V": (B,Ts,D), "out": (B,Tt,D)})


In [ ]:
# Demo 2 — numpy cross attention
import numpy as np
Q = np.array([[1.0, 0.0]])          # target
K = np.array([[1.0,0.0],[0.0,1.0]]) # source
V = np.array([[3.0,0],[0,4.0]])
scores = Q@K.T/np.sqrt(2)
w = np.exp(scores-scores.max()); w/=w.sum()
print(w@V)


In [ ]:
# Demo 3 — analogy to RAG packing
print("RAG retrieves docs as 'memory'; cross-attn reads encoder memory continuously.")


### Try it yourself — Cross Attention

1. Give one example where cross-attention is the right inductive bias.


## Glossary

- **self-attention**: Q,K,V from same sequence
- **cross-attention**: Q from one side, K/V from another


### Workshop drill — Attention Mechanism (1)

Restate each section heading as a single exam-ready sentence.


In [ ]:
# Workshop drill 1 — Attention Mechanism
headings = ['Q, K, V', 'Multi-Head Attention', 'Cross Attention']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Attention Mechanism (2)

Change one hyperparameter/assumption in a demo and predict the effect before running.


In [ ]:
# Workshop drill 2 — Attention Mechanism
print('prediction: ...')
print('observation: ...')
print('delta: ...')


### Workshop drill — Attention Mechanism (3)

List production risks (cost, latency, safety, quality) for this topic.


In [ ]:
# Workshop drill 3 — Attention Mechanism
for r in ['cost','latency','safety','quality']:
    print(f'{r}:')


### Workshop drill — Attention Mechanism (4)

Write a tiny unit-testable helper related to the lesson and assert two cases.


In [ ]:
# Workshop drill 4 — Attention Mechanism
def ok(x):
    return x is not None
assert ok(1) and not ok(None)
print('ok')


### Workshop drill — Attention Mechanism (5)

Sketch an API request/response JSON for a realistic call tied to this topic.


In [ ]:
# Workshop drill 5 — Attention Mechanism
import json
print(json.dumps({'model':'...','input':'...','output':'...'}, indent=2))


### Workshop drill — Attention Mechanism (6)

Compare two design alternatives in a markdown table (fill TODOs).


In [ ]:
# Workshop drill 6 — Attention Mechanism
print('| option | pros | cons |')
print('|--------|------|------|')
print('| A | TODO | TODO |')
print('| B | TODO | TODO |')


## Summary & Key Takeaways

- Attention routes information via Q/K similarity over V
- Multi-head increases pattern capacity
- Cross-attention reads an external memory sequence

### Practice

Implement attention with and without √d scaling; compare weight entropy.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
